# Analyse SL-CR Experiment Results

Objective: compare the Prolog continuous-reasoning strategy (`cr`) with
the full recomputation variant (`base`) using ECLYPSE reporter output.


In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RESULTS_DIR = ROOT / "results"


## Load ECLYPSE CSV Reports

Metrics are written by the ECLYPSE CSV reporter under
`results/<ray-experiment>/<trial>/output/csv/`.


In [ ]:
frames = []
for csv_path in sorted(RESULTS_DIR.glob("**/output/csv/*.csv")):
    frame = pd.read_csv(csv_path)
    frame["trial"] = csv_path.parents[2].name
    frame["report_type"] = csv_path.stem
    frames.append(frame)

reports = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"Loaded {len(reports)} rows")
reports.head()


## Metric Table


In [ ]:
if reports.empty:
    print("No reports yet. Run: uv run python -m sl_cr_experiments.main")
else:
    metrics = reports.pivot_table(
        index=["trial", "n_event"],
        columns="callback_id",
        values="value",
        aggfunc="first",
    ).reset_index()
    for column in metrics.columns:
        if column not in {"trial"}:
            metrics[column] = pd.to_numeric(metrics[column], errors="ignore")
    display(metrics.head())


## Cost Metrics


In [ ]:
if not reports.empty:
    selected = reports[reports["callback_id"].isin(["prolog_query_seconds", "prolog_inferences"])]
    g = sns.catplot(
        data=selected,
        x="callback_id",
        y="value",
        col="trial",
        kind="box",
        sharey=False,
        height=4,
        aspect=1,
    )
    g.set_axis_labels("", "value")


## Repair Impact


In [ ]:
if not reports.empty:
    selected = reports[reports["callback_id"].isin(["placement_migrations", "overloaded_nodes", "placed_components"])]
    g = sns.relplot(
        data=selected,
        x="n_event",
        y="value",
        hue="callback_id",
        col="trial",
        kind="line",
        marker="o",
        facet_kws={"sharey": False},
        height=4,
        aspect=1.2,
    )
    g.set_axis_labels("step", "value")
